# A5 — Preference optimization (DPO)

Synthetic pairs and the Bradley-Terry loss. No Qwen download.
Training and the before/after table stay on the CLI (`README.md`).
Open this notebook from the repo root.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from losses import dpo_sigmoid_loss, rank_by_logprobs
from pairs import build_preference_pairs

items = pd.DataFrame(
    {
        "item_idx": [0, 1, 2, 3],
        "title": ["Portal 2", "", "Halo", "Halo"],
    }
)
cand = pd.DataFrame(
    [
        {
            "user_idx": 0,
            "target_item": 0,
            "history_items": [2],
            "cand_items": [0, 1, 3],
            "cand_scores": [0.0, -1.0, -2.0],
        }
    ]
)
rows = build_preference_pairs(
    cand, items, negatives_per_positive=2, rng=np.random.default_rng(0), max_title_chars=80
)
print(f"{len(rows)} pairs, shared prompt: {rows[0]['prompt'] == rows[1]['prompt']}")
print("--- prompt ---")
print(rows[0]["prompt"])
print("--- chosen / rejected ---")
for row in rows:
    print(repr(row["chosen"]), "||", repr(row["rejected"]), "neg", row["rejected_item"])


The blank title becomes `item 1`. Item 2 (`Halo`) is in the history, so it is not a rejected completion. Item 3 can be, and its title stays `Halo`. If the chosen and rejected strings match, both gain an `[id:]` suffix so DPO is not comparing a string to itself.

The loss below is what `trl.DPOTrainer` minimizes. A zero margin against the reference is `log(2) ≈ 0.693`. A larger policy margin (chosen title more likely than the rejected one, beyond the reference) lowers the loss.

In [ ]:
import math

print(f"log(2) = {math.log(2):.4f}")
print(f"{'policy margin':>14} {'loss':>8}")
for margin in (0.0, 0.5, 1.0, 2.0, 4.0):
    loss = dpo_sigmoid_loss(margin, 0.0, 0.0, 0.0, beta=0.1)
    print(f"{margin:14.1f} {loss:8.4f}")


Ranking uses the mean log-prob of the completion, not the sum. The sum of non-positive token log-probs prefers a short title. Here item 1 is one token with sum −0.20. Item 2 is three tokens with sum −0.45 (mean −0.15). The sum ranks 1 first. The mean ranks 2 first.

In [ ]:
item_ids = [1, 2]
sums = [-0.20, -0.45]
counts = [1, 3]
print("sum rank", rank_by_logprobs(item_ids, sums, counts, length_normalize=False))
print("mean rank", rank_by_logprobs(item_ids, sums, counts, length_normalize=True))


## Reading a real run

```bash
python data/prepare_preferences.py --config configs/dpo_small.yaml
python src/train_dpo.py --config configs/dpo_small.yaml --max_steps 20 --max_pairs 32
python src/eval.py --config configs/dpo_small.yaml --adapter checkpoints/dpo_small
```

`checkpoints/.../test_metrics.json` has three blocks: `retriever` (candidate order), `reference` (base Instruct, or the merged A4 SFT model when `ref_is_sft` is true), and `dpo`. Compare `ndcg@10`, `pair_accuracy`, and `reward_margin`. A margin that rises while NDCG falls is the failure mode in the implementation doc — the policy fit the sampled pairs, not the next item. Fill `RESULTS.md` from that file, including the causal note: the rejected title is a hard negative, not a logged skip.